In [1]:
# Core imports for the whole lab
import numpy as np
import sympy as sp
x, y = sp.symbols('x y')      # symbolic variables we'll reuse
sp.init_printing()            # pretty-print symbolic math
np.random.seed(42)
print('Setup complete. SymPy', sp.__version__, '| NumPy', np.__version__)

Setup complete. SymPy 1.14.0 | NumPy 2.1.3


In [2]:
#1.Derivatives — the slope of a function
#1A. NUMERICAL DERIVATIVE (finite difference)
def f(x):
    return x**2
h = 1e-6
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 6)')
#1B. SYMBOLIC DERIVATIVE (SymPy)
expr = x**2
deriv = sp.diff(expr, x)          # differentiate w.r.t. x
print('d/dx (x**2) =', deriv)     # 2*x
print('Symbolic f\'(3) =', deriv.subs(x, 3))

Numerical f'(3): 6.0  (exact = 6)
d/dx (x**2) = 2*x
Symbolic f'(3) = 6


In [3]:
print("excercise-1")
def g(x):
    return x**3 + 2*x
# 1. Numerical derivative at x = 2
h = 1e-6
numerical_derivative = (g(2 + h) - g(2)) / h
print("Numerical g'(2):", numerical_derivative)
# 2.Symbolic derivative of x**3 + 2*x
expr = x**3 + 2*x
deriv = sp.diff(expr, x)
print("dg/dx =", deriv)
# 3. Evaluate the symbolic derivative at x = 2
symbolic_derivative = deriv.subs(x, 2)
print("Symbolic g'(2):", symbolic_derivative)
# Compare
print("Match:", abs(numerical_derivative - float(symbolic_derivative)) < 1e-5)

excercise-1
Numerical g'(2): 14.000006002490295
dg/dx = 3*x**2 + 2
Symbolic g'(2): 14
Match: True


In [4]:
#2.Partial derivatives & the gradient
#2A. PARTIAL DERIVATIVES
f2 = x**2 + 3*x*y + y**2
print('df/dx =', sp.diff(f2, x))     # 2*x + 3*y
print('df/dy =', sp.diff(f2, y))     # 3*x + 2*y
#2B. THE GRADIENT (vector of partials)
grad = [sp.diff(f2, v) for v in (x, y)]
print('grad f =', grad)
grad_at = [g.subs({x: 1, y: 2}) for g in grad]
print('grad f at (1, 2) =', grad_at)   # points in the steepest-ascent direction

df/dx = 2*x + 3*y
df/dy = 3*x + 2*y
grad f = [2*x + 3*y, 3*x + 2*y]
grad f at (1, 2) = [8, 7]


In [5]:
print("excercise-2")
h2 = x**2 * y + sp.sin(y)
# 1. dh/dx and dh/dy
dh_dx = sp.diff(h2, x)
dh_dy = sp.diff(h2, y)
print("dh/dx =", dh_dx)
print("dh/dy =", dh_dy)
# 2. Assemble the gradient list
grad_h2 = [dh_dx, dh_dy]
print("grad h =", grad_h2)
# 3. Evaluate at (x=2, y=0)
grad_at = [g.subs({x: 2, y: 0}) for g in grad_h2]
print("grad h at (2, 0) =", grad_at)

excercise-2
dh/dx = 2*x*y
dh/dy = x**2 + cos(y)
grad h = [2*x*y, x**2 + cos(y)]
grad h at (2, 0) = [0, 5]


In [6]:
#3.The chain rule
#3A. CHAIN RULE BY HAND vs SymPy
by_hand = sp.cos(x**2) * 2*x
by_sympy = sp.diff(sp.sin(x**2), x)
print('By hand :', by_hand)
print('By SymPy:', by_sympy)
print('Match?  ', sp.simplify(by_hand - by_sympy) == 0)
#3B. CHAINING THREE FUNCTIONS
expr3 = (3*x + 1)**4
print('d/dx (3x+1)^4 =', sp.diff(expr3, x))   # 12*(3x+1)^3

By hand : 2*x*cos(x**2)
By SymPy: 2*x*cos(x**2)
Match?   True
d/dx (3x+1)^4 = 12*(3*x + 1)**3


In [7]:
print("excercise-3")
# y = exp(x**2 + 1); inner = x**2 + 1, inner' = 2x
# 1. By hand (outer derivative × inner derivative)
by_hand = sp.exp(x**2 + 1) * 2*x
print("By hand :", by_hand)
# 2. With sp.diff
by_sympy = sp.diff(sp.exp(x**2 + 1), x)
print("By SymPy:", by_sympy)
# 3. Confirm they match
print("Match?  ", sp.simplify(by_hand - by_sympy) == 0)

excercise-3
By hand : 2*x*exp(x**2 + 1)
By SymPy: 2*x*exp(x**2 + 1)
Match?   True


In [8]:
#4.Backpropagation — the chain rule in a network
#4A. FORWARD PASS
X = np.random.randn(4, 3)
Y = np.random.randn(4, 1)
W1 = np.random.randn(3, 5) * 0.1
W2 = np.random.randn(5, 1) * 0.1
z1 = X @ W1                 # linear layer 1
h  = np.maximum(0, z1)      # ReLU activation
y_hat = h @ W2              # linear layer 2 (prediction)
loss = ((y_hat - Y) ** 2).mean()
print('Initial loss:', round(loss, 4))
#4B. BACKWARD PASS (gradients via the chain rule)
dy   = 2 * (y_hat - Y) / Y.size      # d loss / d y_hat
dW2  = h.T @ dy                      # d loss / d W2
dh   = dy @ W2.T                     # d loss / d h
dz1  = dh * (z1 > 0)                 # ReLU gradient (1 where z1>0 else 0)
dW1  = X.T @ dz1                     # d loss / d W1
print('dW1 shape:', dW1.shape, '(matches W1)')
print('dW2 shape:', dW2.shape, '(matches W2)')
#4C. ONE GRADIENT-DESCENT STEP SHOULD LOWER THE LOSS
lr = 0.1
W1 -= lr * dW1               # step downhill
W2 -= lr * dW2
# Recompute the loss after the update
h_new = np.maximum(0, X @ W1)
loss_new = ((h_new @ W2 - Y) ** 2).mean()
print('Loss before:', round(loss, 4))
print('Loss after :', round(loss_new, 4), '-> should be lower')

Initial loss: 1.6936
dW1 shape: (3, 5) (matches W1)
dW2 shape: (5, 1) (matches W2)
Loss before: 1.6936
Loss after : 1.6524 -> should be lower


In [9]:
print("excercise-4")
Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1     # input -> hidden
Wb = np.random.randn(8, 1) * 0.1     # hidden -> output
# 1. Forward pass
z1 = Xb @ Wa
h = np.maximum(0, z1)                # ReLU
y_hat = h @ Wb
loss = ((y_hat - Yb) ** 2).mean()
# 2. Backward pass
dy = 2 * (y_hat - Yb) / Yb.size
dWb = h.T @ dy
dh = dy @ Wb.T
dz1 = dh * (z1 > 0)                 # ReLU gradient
dWa = Xb.T @ dz1
# 3. One gradient-descent step
lr = 0.05
Wa -= lr * dWa
Wb -= lr * dWb
# Recompute loss after update
h_new = np.maximum(0, Xb @ Wa)
y_hat_new = h_new @ Wb
loss_new = ((y_hat_new - Yb) ** 2).mean()
print("Loss before:", round(loss, 4))
print("Loss after :", round(loss_new, 4))

excercise-4
Loss before: 0.8782
Loss after : 0.8719


In [10]:
#5.Hessian & gradient descent
#5A. THE HESSIAN (matrix of second derivatives = curvature)
f5 = x**2 + 3*x*y + y**2
H = sp.hessian(f5, (x, y))
print('Hessian of f:')
sp.pprint(H)        # [[2, 3], [3, 2]]
#5B. GRADIENT DESCENT ON A SIMPLE FUNCTION
xv = 0.0          # starting point
lr = 0.2
for step in range(15):
    grad = 2 * (xv - 4)        # the derivative
    xv = xv - lr * grad        # step against the gradient
print('Converged x:', round(xv, 3), ' (true minimum = 4)')

Hessian of f:
⎡2  3⎤
⎢    ⎥
⎣3  2⎦
Converged x: 3.998  (true minimum = 4)


In [11]:
print("excercise-5")
# 1. Hessian of x**4 + y**2
f = x**4 + y**2
H = sp.hessian(f, (x, y))
print("Hessian of f:")
sp.pprint(H)
# 2. Gradient descent to minimise (x - 7)**2
xv = 0.0
lr = 0.1
for step in range(20):
    grad = 2 * (xv - 7)
    xv = xv - lr * grad
print("Final x:", xv)

excercise-5
Hessian of f:
⎡    2   ⎤
⎢12⋅x   0⎥
⎢        ⎥
⎣  0    2⎦
Final x: 6.91929549467752
